In [1]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BNF_KMeans") \
    .master("local[*]") \
    .getOrCreate()



practice_bnf_chapter_shares = spark.read.parquet(
    "spark_output/practice_bnf_chapter_shares"
)

26/06/29 12:09:52 WARN Utils: Your hostname, codespaces-475009 resolves to a loopback address: 127.0.0.1; using 10.0.1.232 instead (on interface eth0)
26/06/29 12:09:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 12:09:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
feature_cols = [c for c in practice_bnf_chapter_shares.columns 
                if c != "PRACTICE_CODE"]   

In [11]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

In [12]:
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

In [13]:
kmeans = KMeans(
    k=5,   # change based on elbow/silhouette
    seed=42,
    featuresCol="features"
)

In [14]:
pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model = pipeline.fit(practice_bnf_chapter_shares)

26/06/29 12:26:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/29 12:26:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/29 12:26:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


In [15]:
clustered_df = model.transform(practice_bnf_chapter_shares)

In [16]:
clustered_df.select("practice_code", "prediction").show(20)

+-------------+----------+
|practice_code|prediction|
+-------------+----------+
|            -|         1|
|       00L998|         0|
|       00Q998|         1|
|       00R998|         1|
|       00Y998|         1|
|       01D998|         1|
|       01E998|         0|
|       01F998|         2|
|       01H998|         1|
|       01K998|         0|
|       01V998|         0|
|       01Y998|         1|
|       02A998|         1|
|       02M998|         0|
|       02P998|         0|
|       03F998|         0|
|       03H998|         1|
|       03K998|         0|
|       03W998|         0|
|       04V998|         0|
+-------------+----------+
only showing top 20 rows



In [17]:
cluster_summary = clustered_df.groupBy("prediction").avg(*feature_cols)
cluster_summary.show()

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|prediction|  avg(cost_share_01)|  avg(cost_share_02)|  avg(cost_share_03)|  avg(cost_share_04)|  avg(cos

In [18]:
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

scores = []

for k in [4, 5, 6]:
    
    kmeans = KMeans(
        k=k,
        seed=42,
        featuresCol="features"
    )
    
    pipeline = Pipeline(stages=[assembler, scaler, kmeans])
    model = pipeline.fit(practice_bnf_chapter_shares)
    
    predictions = model.transform(practice_bnf_chapter_shares)
    
    score = evaluator.evaluate(predictions)
    
    scores.append((k, score))

scores

import pandas as pd

pd.DataFrame(scores, columns=["k", "silhouette"]).sort_values("silhouette", ascending=False)

,k,silhouette
2,6,0.696664
1,5,0.479737
0,4,0.450394


In [19]:
kmeans = KMeans(
    k=6,
    seed=42,
    featuresCol="features"
)

pipeline = Pipeline(stages=[assembler, scaler, kmeans])
final_model = pipeline.fit(practice_bnf_chapter_shares)

final_clustered_df = final_model.transform(practice_bnf_chapter_shares)

In [20]:
final_clustered_df.groupBy("prediction").count().orderBy("prediction").show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0| 9565|
|         1|  117|
|         2|   81|
|         3|   88|
|         4|   57|
|         5|  231|
+----------+-----+



In [21]:
final_cluster_profile = final_clustered_df.groupBy("prediction").avg(*feature_cols)
final_cluster_profile.show()

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|prediction|  avg(cost_share_01)|  avg(cost_share_02)|  avg(cost_share_03)|  avg(cost_share_04)|  avg(cos

In [22]:
final_clustered_df.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|         1|  117|
|         3|   88|
|         5|  231|
|         4|   57|
|         2|   81|
|         0| 9565|
+----------+-----+

